# 00: Setup for DNA language model demographic inference

**The angle.** pg-gan-mosquito feeds a naive `(SNPs × haplotypes)` matrix of 0s and 1s to its CNN discriminator. Everything the CNN knows about mosquito biology has to be learned from those bits during training.

This project tests an alternative: **use a pre-trained DNA language model to embed each haplotype into a rich feature representation first, then feed those embeddings to a discriminator instead of the raw 0/1 matrix.** The hypothesis is that DNA-LM embeddings already contain useful biological context (mutation-rate structure, regulatory-element awareness, codon patterns) that the CNN would otherwise have to learn from scratch.

This is directly parallel to how Preston used ESM-2 (a protein language model) to embed protein sequences in the CRC_Inhibitor_ML project. Same recipe, different alphabet: DNA (A, C, G, T) instead of amino acids.

**The DNA-LM we'll start with:** `InstaDeepAI/nucleotide-transformer-v2-500m-multi-species`. 500M parameters, trained on 850 species including non-model organisms. Fits comfortably on a 16 GB GPU for inference.

**Naming note.** The folder is called `Protein-LM` for parity with Preston's prior ESM-2 work, but the actual technique used here operates on nucleotide sequences via a DNA language model.

**What this notebook does.** Sets up paths, imports, and directories. Also reports which packages are installed and which still need to be installed. No science yet — that starts in notebook 01.

**Kernel:** Python 3.11 (Mathieson-LM)

**Venv:** `C:\Users\palla\venvs\Mathieson-LM\` (separate from the pg-gan-mosquito venv, which is TensorFlow-based)

In [ ]:
# ============================================================
# Cell 1: Setup — packages, paths, directories
# ============================================================

import sys
from pathlib import Path

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent                     # Protein-LM/
MATHIESON_APP = PROJECT_ROOT.parent                    # Mathieson_Application/

# Locate pg-gan-mosquito (for shared imports if needed later)
PGGAN_REPO = MATHIESON_APP / "pg-gan-mosquito"
if PGGAN_REPO.exists():
    sys.path.insert(0, str(PGGAN_REPO))
    print(f"pg-gan-mosquito available at: {PGGAN_REPO}")
else:
    print(f"NOTE: pg-gan-mosquito not found at {PGGAN_REPO}")

# Data / models / outputs directories
DATA_DIR       = PROJECT_ROOT / "data"
DATA_RAW       = DATA_DIR / "raw"
DATA_PROCESSED = DATA_DIR / "processed"
MODELS_DIR     = PROJECT_ROOT / "models"
OUTPUTS_DIR    = PROJECT_ROOT / "outputs"

for d in [DATA_RAW, DATA_PROCESSED, MODELS_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"\nDirectory layout:")
print(f"  data/raw:       {DATA_RAW}")
print(f"  data/processed: {DATA_PROCESSED}")
print(f"  models:         {MODELS_DIR}")
print(f"  outputs:        {OUTPUTS_DIR}")

# ------------------------------------------------------------
# Standard ML / data stack
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("Package check")
print("=" * 60)

def check(name, import_as=None, why=None):
    module = import_as or name
    try:
        mod = __import__(module)
        version = getattr(mod, "__version__", "?")
        print(f"  OK   {name:<22} version {version}")
        return True
    except ImportError:
        why_str = f"   ({why})" if why else ""
        print(f"  MISS {name:<22} NOT installed{why_str}")
        return False

print("\nBase scientific Python:")
check("numpy")
check("pandas")
check("matplotlib")
check("scikit-learn", import_as="sklearn")

print("\nPyTorch stack:")
torch_ok = check("torch", why="needed for DNA-LM inference on GPU")

if torch_ok:
    import torch
    print(f"       CUDA available:  {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"       GPU:             {torch.cuda.get_device_name(0)}")
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"       GPU memory:      {vram_gb:.1f} GB")

print("\nHuggingFace stack:")
check("transformers",  why="loads pre-trained DNA-LM checkpoints")
check("accelerate",    why="multi-GPU / mixed-precision loading")
check("huggingface_hub", why="downloading models")

print("\nGenomic data tools:")
check("pyfaidx",         why="random-access to reference genome FASTA")
check("pysam",           why="VCF/BAM handling if needed")
check("scikit-allel",    import_as="allel", why="reading Ag1000G HDF5 files")
check("malariagen_data", why="direct Python access to Ag1000G Phase 3")
check("biopython",       import_as="Bio", why="sequence manipulation utilities")

print("\n" + "=" * 60)
print("Setup complete.")
print("=" * 60)